In [1]:
from src.run_app import *
from src.prepare_gam import *
from src.utils import *
from gam_rs_utils.binarize_dataset import binarize_dataset
from gam_rs_utils.utils import *
from FasterRisk.src.fasterrisk import fasterrisk
from time import time
import pickle

dataset_settings = {
    'bank': {
        'starting_gap_tolerance': 0.0015,
        'gap_tolerance_interval': 0.0015,
        'num_estimators': 50,
    },
    'compas': {
        'starting_gap_tolerance': 0.0005,
        'gap_tolerance_interval': 0.0005,
        'num_estimators': 50,
    },
    'diabetes': {
        'starting_gap_tolerance': 0.001,
        'gap_tolerance_interval': 0.001,
        'num_estimators': 200,
    },
    # 'heloc_original'
    # 'hiv'
    'netherlands': {
        'starting_gap_tolerance': 0.0005,
        'gap_tolerance_interval': 0.0005,
        'num_estimators': 50,
    },
    'spambase': {
        'starting_gap_tolerance': 0.003,
        'gap_tolerance_interval': 0.0005,
        'num_estimators': 50,
    },
}

def run_tolerance_tests(use_beam, params, file_name):
    tries = 6
    results = []
    for dataset_name, settings in dataset_settings.items():
        ne = settings["num_estimators"]

        path = 'datasets/{}.csv'.format(dataset_name)
        dataset = pd.read_csv(path)
        print(f"Dataset: {dataset_name}")
        print(f"Binarized shape: {dataset.shape}")

        df, thresholds, header, threshold_guess_time = binarize_dataset(dataset, ne)
        X, y = df.iloc[:, :-1], df.iloc[:, -1]
        header = pd.Index(["intercept"] + list(X.columns)).astype("object")
        X_one_hot, y = utils.get_X_y(X, y)

        for i in range(tries):
            gt = settings['starting_gap_tolerance'] + i * settings['gap_tolerance_interval']

            start = time()
            rs = fasterrisk.RiskScoreOptimizer(X_one_hot, y, k=10, lb=-100, ub=100, gap_tolerance=gt, select_top_m=-1, maxAttempts=25)
            if use_beam:
                rs.optimize_with_swaps_beam_search(**params)
            else:
                rs.optimize_with_swaps(**params)
            end = time()

            result = {
                "dataset": dataset_name,
                "dataset_shape": dataset.shape,
                "num_estimators": ne,
                "gap_tolerance": gt,
                "feature_selection": "top",
                "threshold_guess_time": threshold_guess_time,
                "num_features": len(header),
                "runtime": end - start,
                "betas": rs.sparseDiversePool_betas,
                "beta0": rs.sparseDiversePool_beta0,
                "num_solutions": rs.sparseDiversePool_betas.shape[0],
                "loss": get_loss(X_one_hot, y, rs.sparseDiversePool_beta0, rs.sparseDiversePool_betas)
            }
            results.append(result)
            print(f"\t{gt} tolerance, {rs.sparseDiversePool_betas.shape[0]} solutions, {end - start:.2f} seconds")

            if end - start > 120:
                print("Last optimization took more than 120 seconds, stopping further tries.")
                break

    with open(f"results/{file_name}.pkl", "wb") as f:
        pickle.dump(results, f)

In [2]:
run_tolerance_tests(
    False,
    {"swaps": 3, "fanout_decay": 1, "feature_selection": "top"},
    "tolerance"
)

Dataset: bank
Binarized shape: (4521, 17)
	0.0015 tolerance, 0 solutions, 25.66 seconds
	0.003 tolerance, 0 solutions, 19.57 seconds
	0.0045000000000000005 tolerance, 3 solutions, 34.36 seconds
	0.006 tolerance, 52 solutions, 48.82 seconds
	0.0075 tolerance, 179 solutions, 73.14 seconds
	0.009 tolerance, 477 solutions, 159.00 seconds
Last optimization took more than 120 seconds, stopping further tries.
Dataset: compas
Binarized shape: (6907, 8)
	0.0005 tolerance, 2 solutions, 14.74 seconds
	0.001 tolerance, 14 solutions, 19.31 seconds
	0.0015 tolerance, 41 solutions, 22.77 seconds
	0.002 tolerance, 126 solutions, 36.76 seconds
	0.0025 tolerance, 235 solutions, 63.43 seconds
	0.003 tolerance, 414 solutions, 94.50 seconds
Dataset: diabetes
Binarized shape: (768, 9)
	0.001 tolerance, 10 solutions, 2.82 seconds
	0.002 tolerance, 18 solutions, 3.26 seconds
	0.003 tolerance, 31 solutions, 3.86 seconds
	0.004 tolerance, 53 solutions, 4.51 seconds
	0.005 tolerance, 81 solutions, 6.02 seconds
	

In [3]:
run_tolerance_tests(
    True,
    {"swaps": 3, "beam_size": 200},
    "tolerance_beam"
)

Dataset: bank
Binarized shape: (4521, 17)
	0.0015 tolerance, 0 solutions, 17.53 seconds
	0.003 tolerance, 0 solutions, 18.32 seconds
	0.0045000000000000005 tolerance, 3 solutions, 20.23 seconds
	0.006 tolerance, 52 solutions, 27.13 seconds
	0.0075 tolerance, 36 solutions, 52.31 seconds
	0.009 tolerance, 35 solutions, 89.87 seconds
Dataset: compas
Binarized shape: (6907, 8)
	0.0005 tolerance, 2 solutions, 13.36 seconds
	0.001 tolerance, 14 solutions, 16.32 seconds
	0.0015 tolerance, 41 solutions, 21.09 seconds
	0.002 tolerance, 34 solutions, 34.80 seconds
	0.0025 tolerance, 26 solutions, 58.40 seconds
	0.003 tolerance, 24 solutions, 75.75 seconds
Dataset: diabetes
Binarized shape: (768, 9)
	0.001 tolerance, 10 solutions, 3.04 seconds
	0.002 tolerance, 18 solutions, 3.82 seconds
	0.003 tolerance, 31 solutions, 4.53 seconds
	0.004 tolerance, 53 solutions, 5.24 seconds
	0.005 tolerance, 43 solutions, 7.39 seconds
	0.006 tolerance, 37 solutions, 9.93 seconds
Dataset: netherlands
Binarized s